In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install kagglehub


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("aliabdelmenam/rdd-2022")
print("Dataset path:", dataset_path)

Using Colab cache for faster access to the 'rdd-2022' dataset.
Dataset path: /kaggle/input/rdd-2022


In [ ]:
from pathlib import Path

BASE_DIR = Path(dataset_path)

print("BASE_DIR:", BASE_DIR)
print("Contents of BASE_DIR:")
for p in sorted(BASE_DIR.iterdir()):
    print("-", p.name)

BASE_DIR: /kaggle/input/rdd-2022
Contents of BASE_DIR:
- RDD_SPLIT


In [ ]:
DATA_DIR = BASE_DIR / "RDD_SPLIT"

print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    print("Contents:")
    for p in sorted(DATA_DIR.iterdir()):
        print("-", p.name)

DATA_DIR: /kaggle/input/rdd-2022/RDD_SPLIT
DATA_DIR exists: True
Contents:
- test
- train
- val


## Dataset Count Check
Count images and label files in train, val, and test.

In [ ]:
from glob import glob

splits = ["train", "val", "test"]
image_suffixes = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]

def count_images(image_dir):
    files = []
    for ext in image_suffixes:
        files.extend(glob(str(image_dir / ext)))
    return len(files)

for split in splits:
    image_dir = DATA_DIR / split / "images"
    label_dir = DATA_DIR / split / "labels"

    num_images = count_images(image_dir)
    num_labels = len(glob(str(label_dir / "*.txt")))

    print(f"{split.upper()}:")
    print(f"  Images: {num_images}")
    print(f"  Labels: {num_labels}")

TRAIN:
  Images: 26869
  Labels: 26869
VAL:
  Images: 5758
  Labels: 5758
TEST:
  Images: 5758
  Labels: 5758


## Quick Label Validation
Scan all label files and count empty labels, invalid formats, and invalid bounding boxes.

In [ ]:
all_label_files = list((DATA_DIR / "train" / "labels").glob("*.txt")) \
                + list((DATA_DIR / "val" / "labels").glob("*.txt")) \
                + list((DATA_DIR / "test" / "labels").glob("*.txt"))

empty_files = []
bad_format = []
bad_bbox = []
class_ids_found = set()
total_boxes = 0

for label_file in all_label_files:
    lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]

    if len(lines) == 0:
        empty_files.append(label_file)
        continue

    for line_num, line in enumerate(lines, start=1):
        parts = line.split()

        if len(parts) != 5:
            bad_format.append((label_file.name, line_num, line))
            continue

        try:
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
        except:
            bad_format.append((label_file.name, line_num, line))
            continue

        class_ids_found.add(class_id)
        total_boxes += 1

        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
            bad_bbox.append((label_file.name, line_num, (x_center, y_center, width, height)))

print("Total label files:", len(all_label_files))
print("Empty label files:", len(empty_files))
print("Bad format lines:", len(bad_format))
print("Bad bbox values:", len(bad_bbox))
print("Total boxes:", total_boxes)
print("Class IDs found:", sorted(class_ids_found))

if bad_bbox[:5]:
    print("\nSample bad bbox:")
    for item in bad_bbox[:5]:
        print(item)

Total label files: 38385
Empty label files: 11724
Bad format lines: 0
Bad bbox values: 1
Total boxes: 65712
Class IDs found: [0, 1, 2, 3, 4]

Sample bad bbox:
('Japan_001265.txt', 4, (0.33, 0.790833, 0.0, 0.001667))


## Create Writable Working Copy
Copy the dataset from the read-only Kaggle path into a writable Colab working directory.


In [ ]:
import shutil
from pathlib import Path

WORK_DIR = Path("/content/RDD_WORK")
WORK_DATA_DIR = WORK_DIR / "RDD_SPLIT"

if WORK_DATA_DIR.exists():
    shutil.rmtree(WORK_DATA_DIR)

shutil.copytree(DATA_DIR, WORK_DATA_DIR)

print("Original DATA_DIR:", DATA_DIR)
print("Writable WORK_DATA_DIR:", WORK_DATA_DIR)
print("WORK_DATA_DIR exists:", WORK_DATA_DIR.exists())

for p in sorted(WORK_DATA_DIR.iterdir()):
    print("-", p.name)

Original DATA_DIR: /kaggle/input/rdd-2022/RDD_SPLIT
Writable WORK_DATA_DIR: /content/RDD_WORK/RDD_SPLIT
WORK_DATA_DIR exists: True
- test
- train
- val


## Fix Invalid Bounding Box
Remove the invalid YOLO annotation line from the writable training label file.

In [ ]:
target_file = WORK_DATA_DIR / "train" / "labels" / "Japan_001265.txt"

lines = target_file.read_text().splitlines()
cleaned_lines = []
removed_lines = []

for i, line in enumerate(lines, start=1):
    line = line.strip()
    if not line:
        continue

    parts = line.split()

    if len(parts) != 5:
        removed_lines.append((i, line, "bad_format"))
        continue

    try:
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])
    except:
        removed_lines.append((i, line, "parse_error"))
        continue

    is_valid = (
        class_id >= 0 and
        0 <= x_center <= 1 and
        0 <= y_center <= 1 and
        0 < width <= 1 and
        0 < height <= 1
    )

    if is_valid:
        cleaned_lines.append(line)
    else:
        removed_lines.append((i, line, "invalid_bbox"))

target_file.write_text("\n".join(cleaned_lines) + ("\n" if cleaned_lines else ""))

print("File cleaned:", target_file.name)
print("Removed lines:", len(removed_lines))
print("Remaining valid lines:", len(cleaned_lines))

if removed_lines:
    print("\nRemoved entries:")
    for item in removed_lines:
        print(item)

File cleaned: Japan_001265.txt
Removed lines: 1
Remaining valid lines: 3

Removed entries:
(4, '2 0.330000 0.790833 0.000000 0.001667', 'invalid_bbox')


## Re-check Bounding Boxes
Verify that no invalid YOLO bounding boxes remain after cleaning.

In [ ]:
bad_bbox_after_cleaning = []

for split in splits:
    for label_file in (WORK_DATA_DIR / split / "labels").glob("*.txt"):
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]

        for line_num, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                continue

            try:
                class_id = int(parts[0])
                x_center, y_center, width, height = map(float, parts[1:])
            except:
                continue

            if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
                bad_bbox_after_cleaning.append((label_file.name, line_num, (x_center, y_center, width, height)))

print("Invalid bbox entries after cleaning:", len(bad_bbox_after_cleaning))

if bad_bbox_after_cleaning[:5]:
    for item in bad_bbox_after_cleaning[:5]:
        print(item)

Invalid bbox entries after cleaning: 0


## Create YOLO Image Lists
Generate train, val, and test image path lists from the cleaned writable dataset.

In [ ]:
YOLO_DIR = Path("/content/YOLO_READY")
YOLO_DIR.mkdir(parents=True, exist_ok=True)

split_list_files = {}

for split in splits:
    image_paths = []

    image_dir = WORK_DATA_DIR / split / "images"
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            image_paths.append(str(image_path.resolve()))

    list_file = YOLO_DIR / f"{split}.txt"
    list_file.write_text("\n".join(image_paths) + "\n", encoding="utf-8")
    split_list_files[split] = list_file

    print(f"{split}:")
    print(f"  images listed: {len(image_paths)}")
    print(f"  file: {list_file}")

train:
  images listed: 26869
  file: /content/YOLO_READY/train.txt
val:
  images listed: 5758
  file: /content/YOLO_READY/val.txt
test:
  images listed: 5758
  file: /content/YOLO_READY/test.txt


## Install Ultralytics
Install the YOLO training package and verify the installed version.


In [ ]:
!pip -q install -U ultralytics

from ultralytics import YOLO
import ultralytics

print("Ultralytics version:", ultralytics.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.4.33


## Create 3-Class Label Version
Create a new writable dataset version where all crack subclasses are merged into one class.

In [ ]:
import shutil
from pathlib import Path

MERGED_WORK_DIR = Path("/content/RDD_WORK_3CLASS")
MERGED_DATA_DIR = MERGED_WORK_DIR / "RDD_SPLIT"

# Start from a fresh copy
if MERGED_DATA_DIR.exists():
    shutil.rmtree(MERGED_DATA_DIR)

shutil.copytree(WORK_DATA_DIR, MERGED_DATA_DIR)

# Mapping:
# 0,1,2 -> 0 (crack)
# 3     -> 1 (other corruption)
# 4     -> 2 (Pothole)
class_map = {
    0: 0,
    1: 0,
    2: 0,
    3: 1,
    4: 2,
}

changed_files = 0
changed_lines = 0

for split in splits:
    label_dir = MERGED_DATA_DIR / split / "labels"

    for label_file in label_dir.glob("*.txt"):
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
        new_lines = []

        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                continue

            old_class = int(parts[0])
            new_class = class_map[old_class]

            if new_class != old_class:
                changed_lines += 1

            new_line = " ".join([str(new_class)] + parts[1:])
            new_lines.append(new_line)

        label_file.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))

        if lines != new_lines:
            changed_files += 1

print("MERGED_DATA_DIR:", MERGED_DATA_DIR)
print("Changed files:", changed_files)
print("Changed label lines:", changed_lines)
print("Example label path:", MERGED_DATA_DIR / "train" / "labels" / "Japan_001265.txt")

MERGED_DATA_DIR: /content/RDD_WORK_3CLASS/RDD_SPLIT
Changed files: 20185
Changed label lines: 39695
Example label path: /content/RDD_WORK_3CLASS/RDD_SPLIT/train/labels/Japan_001265.txt


## Verify 3-Class Labels
Check that the merged dataset now contains only the new class IDs: 0, 1, and 2.

In [ ]:
class_ids_found = set()
total_boxes = 0

for split in splits:
    for label_file in (MERGED_DATA_DIR / split / "labels").glob("*.txt"):
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]

        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            class_ids_found.add(class_id)
            total_boxes += 1

print("Class IDs found:", sorted(class_ids_found))
print("Total boxes:", total_boxes)

Class IDs found: [0, 1, 2]
Total boxes: 65711


## Find Pothole Images in 3-Class Dataset
Identify all training images that contain class 2 (Pothole) in the merged 3-class dataset.

In [ ]:
POTHOLE_CLASS_ID = 2

train_image_dir = MERGED_DATA_DIR / "train" / "images"
train_label_dir = MERGED_DATA_DIR / "train" / "labels"

all_train_images_3class = []
pothole_images_3class = []

for image_path in sorted(train_image_dir.iterdir()):
    if image_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue

    all_train_images_3class.append(str(image_path.resolve()))

    label_path = train_label_dir / f"{image_path.stem}.txt"
    if not label_path.exists():
        continue

    lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    has_pothole = any(int(line.split()[0]) == POTHOLE_CLASS_ID for line in lines if len(line.split()) == 5)

    if has_pothole:
        pothole_images_3class.append(str(image_path.resolve()))

print("Total train images:", len(all_train_images_3class))
print("Train images with pothole:", len(pothole_images_3class))
print("Pothole image ratio:", f"{len(pothole_images_3class) / len(all_train_images_3class):.2%}")

print("\nSample pothole images:")
for p in pothole_images_3class[:5]:
    print(p)

Total train images: 26869
Train images with pothole: 2599
Pothole image ratio: 9.67%

Sample pothole images:
/content/RDD_WORK_3CLASS/RDD_SPLIT/train/images/China_Drone_000048.jpg
/content/RDD_WORK_3CLASS/RDD_SPLIT/train/images/China_Drone_000057.jpg
/content/RDD_WORK_3CLASS/RDD_SPLIT/train/images/China_Drone_000062.jpg
/content/RDD_WORK_3CLASS/RDD_SPLIT/train/images/China_Drone_000064.jpg
/content/RDD_WORK_3CLASS/RDD_SPLIT/train/images/China_Drone_000136.jpg


## Create Pothole-Boosted Train List (3-Class)
Build a new training list by repeating images that contain potholes in the merged 3-class dataset.

In [ ]:
BOOST_FACTOR = 2  # pothole images will appear 2 extra times

boosted_train_images_3class = all_train_images_3class.copy()

for image_path in pothole_images_3class:
    boosted_train_images_3class.extend([image_path] * BOOST_FACTOR)

BOOSTED_TRAIN_TXT_3CLASS = YOLO_DIR / "train_pothole_boosted_3class.txt"
BOOSTED_TRAIN_TXT_3CLASS.write_text("\n".join(boosted_train_images_3class) + "\n", encoding="utf-8")

print("Original train images:", len(all_train_images_3class))
print("Pothole images:", len(pothole_images_3class))
print("Boost factor:", BOOST_FACTOR)
print("Boosted train images:", len(boosted_train_images_3class))
print("Saved to:", BOOSTED_TRAIN_TXT_3CLASS)

Original train images: 26869
Pothole images: 2599
Boost factor: 2
Boosted train images: 32067
Saved to: /content/YOLO_READY/train_pothole_boosted_3class.txt


## Create Pothole-Boosted 3-Class Dataset YAML
Create a YOLO dataset config for the 3-class merged dataset using the pothole-boosted training list.

In [ ]:
import yaml

class_names_3class = [
    "crack",
    "other corruption",
    "Pothole"
]

BOOSTED_DATASET_YAML_3CLASS = YOLO_DIR / "dataset_pothole_boosted_3class.yaml"

boosted_dataset_config_3class = {
    "path": str(YOLO_DIR.resolve()),
    "train": str(BOOSTED_TRAIN_TXT_3CLASS.resolve()),
    "val": str((YOLO_DIR / "val.txt").resolve()),
    "test": str((YOLO_DIR / "test.txt").resolve()),
    "nc": len(class_names_3class),
    "names": class_names_3class
}

with open(BOOSTED_DATASET_YAML_3CLASS, "w", encoding="utf-8") as f:
    yaml.safe_dump(boosted_dataset_config_3class, f, sort_keys=False, allow_unicode=True)

print("BOOSTED_DATASET_YAML_3CLASS created at:")
print(BOOSTED_DATASET_YAML_3CLASS)
print("\nContents:")
print(BOOSTED_DATASET_YAML_3CLASS.read_text(encoding="utf-8"))

BOOSTED_DATASET_YAML_3CLASS created at:
/content/YOLO_READY/dataset_pothole_boosted_3class.yaml

Contents:
path: /content/YOLO_READY
train: /content/YOLO_READY/train_pothole_boosted_3class.txt
val: /content/YOLO_READY/val.txt
test: /content/YOLO_READY/test.txt
nc: 3
names:
- crack
- other corruption
- Pothole



## Rebuild YOLO Lists for the 3-Class Dataset
Create new train, val, and test image lists that point to the merged 3-class dataset.

In [ ]:
YOLO_DIR_3CLASS = Path("/content/YOLO_READY_3CLASS")
YOLO_DIR_3CLASS.mkdir(parents=True, exist_ok=True)

split_list_files_3class = {}

for split in splits:
    image_paths = []
    image_dir = MERGED_DATA_DIR / split / "images"

    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            image_paths.append(str(image_path.resolve()))

    list_file = YOLO_DIR_3CLASS / f"{split}.txt"
    list_file.write_text("\n".join(image_paths) + "\n", encoding="utf-8")
    split_list_files_3class[split] = list_file

    print(f"{split}:")
    print(f"  images listed: {len(image_paths)}")
    print(f"  file: {list_file}")

train:
  images listed: 26869
  file: /content/YOLO_READY_3CLASS/train.txt
val:
  images listed: 5758
  file: /content/YOLO_READY_3CLASS/val.txt
test:
  images listed: 5758
  file: /content/YOLO_READY_3CLASS/test.txt


## Create 3-Class Pothole-Boosted Train List
Create the boosted training list inside the 3-class YOLO directory using the merged dataset paths.

In [ ]:
BOOST_FACTOR = 2

boosted_train_images_3class = all_train_images_3class.copy()
for image_path in pothole_images_3class:
    boosted_train_images_3class.extend([image_path] * BOOST_FACTOR)

BOOSTED_TRAIN_TXT_3CLASS = YOLO_DIR_3CLASS / "train_pothole_boosted_3class.txt"
BOOSTED_TRAIN_TXT_3CLASS.write_text("\n".join(boosted_train_images_3class) + "\n", encoding="utf-8")

print("Original train images:", len(all_train_images_3class))
print("Pothole images:", len(pothole_images_3class))
print("Boost factor:", BOOST_FACTOR)
print("Boosted train images:", len(boosted_train_images_3class))
print("Saved to:", BOOSTED_TRAIN_TXT_3CLASS)

Original train images: 26869
Pothole images: 2599
Boost factor: 2
Boosted train images: 32067
Saved to: /content/YOLO_READY_3CLASS/train_pothole_boosted_3class.txt


## Create 3-Class Pothole-Boosted Dataset YAML
Create the final YOLO dataset configuration for the merged 3-class dataset with pothole boosting.

In [ ]:
import yaml

class_names_3class = [
    "crack",
    "other corruption",
    "Pothole"
]

BOOSTED_DATASET_YAML_3CLASS = YOLO_DIR_3CLASS / "dataset_pothole_boosted_3class.yaml"

boosted_dataset_config_3class = {
    "path": str(YOLO_DIR_3CLASS.resolve()),
    "train": str((YOLO_DIR_3CLASS / "train_pothole_boosted_3class.txt").resolve()),
    "val": str((YOLO_DIR_3CLASS / "val.txt").resolve()),
    "test": str((YOLO_DIR_3CLASS / "test.txt").resolve()),
    "nc": len(class_names_3class),
    "names": class_names_3class
}

with open(BOOSTED_DATASET_YAML_3CLASS, "w", encoding="utf-8") as f:
    yaml.safe_dump(boosted_dataset_config_3class, f, sort_keys=False, allow_unicode=True)

print("BOOSTED_DATASET_YAML_3CLASS created at:")
print(BOOSTED_DATASET_YAML_3CLASS)
print("\nContents:")
print(BOOSTED_DATASET_YAML_3CLASS.read_text(encoding="utf-8"))

BOOSTED_DATASET_YAML_3CLASS created at:
/content/YOLO_READY_3CLASS/dataset_pothole_boosted_3class.yaml

Contents:
path: /content/YOLO_READY_3CLASS
train: /content/YOLO_READY_3CLASS/train_pothole_boosted_3class.txt
val: /content/YOLO_READY_3CLASS/val.txt
test: /content/YOLO_READY_3CLASS/test.txt
nc: 3
names:
- crack
- other corruption
- Pothole



## Train Final 3-Class Pothole-Boosted Model
Train YOLO26m on the merged 3-class dataset with pothole boosting, realistic augmentation, Google Drive checkpointing, and early stopping.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

DRIVE_RUNS_DIR = Path("/content/drive/MyDrive/SABIQ_runs")
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME_FINAL = "yolo26m.pt"
RUN_NAME_FINAL = "yolo26m_3class_pothole_boosted_final"

model_final = YOLO(MODEL_NAME_FINAL)

results_final = model_final.train(
    data=str(BOOSTED_DATASET_YAML_3CLASS),
    imgsz=640,
    epochs=100,
    patience=20,
    batch=16,
    device=0,
    optimizer="auto",
    cos_lr=True,
    close_mosaic=10,
    amp=True,
    cache=False,
    workers=2,
    seed=42,

    project=str(DRIVE_RUNS_DIR),
    name=RUN_NAME_FINAL,
    exist_ok=True,
    save=True,
    save_period=5,
    verbose=True,

    # realistic but slightly stronger augmentation
    degrees=2.0,
    translate=0.06,
    scale=0.20,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    hsv_h=0.01,
    hsv_s=0.45,
    hsv_v=0.30,
    mosaic=0.35,
    mixup=0.0,
    copy_paste=0.0
)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/YOLO_READY_3CLASS/dataset_pothole_boosted_3class.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.45, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=0.35, multi_scale=0.0, name=yolo26m_3class_pothole_boosted_final, nbs=64, nms=False, opset=None, op